In [1]:
# Install pip packages
!mkdir ~/tmpdir
!TMPDIR=~/tmpdir python3 -m pip install --upgrade --no-cache-dir "sagemaker<3" pandas numpy matplotlib seaborn scikit-learn "torch==2.6" torchvision boto3

mkdir: cannot create directory ‘/home/ec2-user/tmpdir’: File exists
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 132.8 MB/s  0:00:030:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 290.2 MB/s  0:00:02eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 254.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 161.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 594.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 138.4 MB/s  0:00:040:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 129.4 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [18]:
# Prepare dataset
%cd ~/SageMaker/
! rm -rf dataset/ dataset.zip
!wget -nc -q -O "dataset.zip" "https://www.kaggle.com/api/v1/datasets/download/ayush1220/cifar10"

!mkdir dataset -p
!unzip -q -o ./dataset.zip -d ./dataset/
!mv ./dataset/cifar10/* ./dataset/
!rm -r ./dataset/cifar10 dataset.zip

%cd ./dataset/
!zip -rq ../dataset.zip .

%cd ..

/home/ec2-user/SageMaker
/home/ec2-user/SageMaker/dataset
/home/ec2-user/SageMaker


In [3]:
# Import pip packages
import pandas as pd
import numpy as np
import torch
import os
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from IPython.display import display
from sagemaker.inputs import TrainingInput
from sagemaker.s3 import S3Uploader

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [4]:
# Set up SageMaker environment
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "sagemaker"

from sagemaker import get_execution_role

role = get_execution_role()

In [23]:
# Upload dataset zip file to S3
inputs = S3Uploader.upload("dataset.zip", "s3://{}/{}".format(bucket, prefix))

In [8]:
# Find mean, std
_dataset = datasets.ImageFolder(root='./dataset/train', transform=transforms.ToTensor())
_loader  = DataLoader(_dataset)

mean   = torch.zeros(3)
sq_mean = torch.zeros(3)
n_pixels = 0

with torch.no_grad():
    for images, _ in _loader:
        b, c, h, w = images.shape
        n_pixels += b * h * w
        mean     += images.sum(dim=[0, 2, 3])
        sq_mean  += (images ** 2).sum(dim=[0, 2, 3])

mean  /= n_pixels
std    = (sq_mean / n_pixels - mean ** 2).sqrt()

mean, std

(tensor([0.4914, 0.4822, 0.4465]), tensor([0.2470, 0.2435, 0.2616]))

In [24]:
from sagemaker.pytorch import PyTorch

estimator = PyTorch(
    entry_point="train.py",
    framework_version="2.6",
    py_version="py312",
    instance_type="ml.m6i.xlarge",
    instance_count=1,
    output_path="s3://{}/{}/output/".format(bucket, prefix),
    checkpoint_s3_uri="s3://{}/{}/checkpoint/".format(bucket, prefix),
    role=role,
    hyperparameters = {
        'epochs': 20,
        'batch_size': 64,
        'lr': 0.001,
        # 'backend': 'gloo'  # distributed
    },
    metric_definitions    = [
        {"Name": "train:loss", "Regex": r"Train Loss: ([0-9\.]+)"},
        {"Name": "train:acc",  "Regex": r"Train Acc: ([0-9\.]+)"},
        {"Name": "val:loss",   "Regex": r"Test Loss: ([0-9\.]+)"},
        {"Name": "val:acc",    "Regex": r"Test Acc: ([0-9\.]+)"},
    ],
)

estimator.fit({
    "train": TrainingInput("s3://{}/{}/dataset.zip".format(bucket, prefix)),  # /opt/ml/input/data/train
})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: pytorch-training-2026-07-01-08-49-50-931


2026-07-01 08:49:55 Starting - Starting the training job...
2026-07-01 08:50:09 Starting - Preparing the instances for training...
2026-07-01 08:50:29 Downloading - Downloading input data...
2026-07-01 08:50:59 Downloading - Downloading the training image...
2026-07-01 08:51:44 Training - Training image download completed. Training in progress.bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2026-07-01 08:51:46,778 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-07-01 08:51:46,778 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-07-01 08:51:46,779 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-07-01 08:51:46,789 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2026-07-01 08:51:46,790 sagemaker_pytorch_container.training INFO     Invoking user training s

In [26]:
# Create new model + Update Endpoint
import boto3
import time

sm = boto3.client("sagemaker")

model = estimator.create_model(
    name = f'cifar10-model-{int(time.time())}'
)
model_name = model.name
model._create_sagemaker_model(instance_type="ml.g5.xlarge")

new_config = f'endpoint-configuration-{int(time.time())}'
sm.create_endpoint_config(
    EndpointConfigName=new_config,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": "ml.g5.xlarge"
    }]
)

INFO:sagemaker:Repacking model artifact (s3://sagemaker-us-east-1-200148130345/sagemaker/output/pytorch-training-2026-07-01-08-49-50-931/output/model.tar.gz), script artifact (s3://sagemaker-us-east-1-200148130345/pytorch-training-2026-07-01-08-49-50-931/source/sourcedir.tar.gz), and dependencies ([]) into single tar.gz file located at s3://sagemaker-us-east-1-200148130345/cifar10-model-1782898848/model.tar.gz. This may take some time depending on model size...
INFO:sagemaker:Creating model with name: cifar10-model-1782898848


{'EndpointConfigArn': 'arn:aws:sagemaker:us-east-1:200148130345:endpoint-config/endpoint-configuration-1782898852',
 'ResponseMetadata': {'RequestId': '7c2a647c-ddb9-4447-acfd-65b6d9dc2fb0',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '7c2a647c-ddb9-4447-acfd-65b6d9dc2fb0',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '114',
   'date': 'Wed, 01 Jul 2026 09:40:52 GMT'},
  'RetryAttempts': 0}}

In [27]:
sm.create_endpoint(
    EndpointName = "cifar10-endpoint",
    EndpointConfigName = new_config
)

# sm.update_endpoint(
#     EndpointName = "cifar10-endpoint",
#     EndpointConfigName = new_config
# )

{'EndpointArn': 'arn:aws:sagemaker:us-east-1:200148130345:endpoint/cifar10-endpoint',
 'ResponseMetadata': {'RequestId': '0cf18a21-0496-4c18-8e87-8a23393eeb92',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '0cf18a21-0496-4c18-8e87-8a23393eeb92',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '84',
   'date': 'Wed, 01 Jul 2026 09:41:09 GMT'},
  'RetryAttempts': 0}}

In [29]:
CLASSES = _dataset.classes
CLASSES

['airplane',
 'automobile',
 'bird',
 'cat',
 'deer',
 'dog',
 'frog',
 'horse',
 'ship',
 'truck']

In [30]:
# Inference test
import base64, json, boto3

client = boto3.client("sagemaker-runtime")
endpoint_name = 'cifar10-endpoint'
image_path = "0004.png"

with open(image_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")
response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   #           application/json | text/plain | image/png
    Body=json.dumps({"image": b64}),  # json.dumps({"image": b64}) |        b64 |  f.read()
)

body = json.load(response["Body"])
print(body)
print(CLASSES[body["class_idx"]])

{'class_idx': 2, 'confidence': 0.7463216781616211}
bird
